# lesson10 learning｜Selenium Notebook

先從上到下執行設定和函式定義格。每次示範或檢查會啟動練習站及 Chrome，結束時會關閉；請等待執行完成後再執行下一格。


In [ ]:

import os
import sys
import time
from pathlib import Path

import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson10_scroll_and_load_more'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())
sys.path.insert(0, str(Path.cwd().parents[1] / "tools"))

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

from practice_site import start_server, stop_server

HEADLESS = os.environ.get("HEADLESS") == "1"




## new_driver


In [ ]:
def new_driver():
    """建立一個 Chrome。視窗大小固定，因為「捲到底才載入」和視窗高度有關。"""
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless=new")  # 不顯示視窗
    options.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=options)


## title


In [ ]:
def title(text):
    print(f"\n=== {text} ===")


## post_count


In [ ]:
def post_count(driver):
    """目前頁面上有幾篇文章。"""
    return len(driver.find_elements(By.CSS_SELECTOR, "article.post"))


## scroll_to_bottom


In [ ]:
def scroll_to_bottom(driver):
    """用 JavaScript 把頁面捲到最底。"""
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")


## parse_posts


In [ ]:
def parse_posts(html):
    """把『已經載入完成』的 HTML 解析成 list of dict。"""
    soup = BeautifulSoup(html, "html.parser")
    return [
        {
            "id": int(article["data-id"]),
            "title": article.select_one("h3.post-title").get_text(strip=True),
            "tag": article.select_one("span.post-tag").get_text(strip=True),
            "views": int(article.select_one("span.post-views").get_text(strip=True)),
        }
        for article in soup.select("article.post")
    ]


## demo_01_preview


In [ ]:
def demo_01_preview(driver, base_url):
    title("01｜今天的成果預覽")
    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))
    print("剛打開網頁：", post_count(driver), "篇")
    for round_number in range(1, 4):
        before = post_count(driver)
        scroll_to_bottom(driver)
        WebDriverWait(driver, 5).until(lambda d: post_count(d) > before)
        print(f"捲到底 {round_number} 次後：", post_count(driver), "篇")
    print("→ 每次捲到底，網頁就再載入 10 篇。今天要讓程式自己捲到全部載入。")


## demo_02_requests_cannot


In [ ]:
def demo_02_requests_cannot(driver, base_url):
    title("02｜requests 只看到空殼")
    feed_html = requests.get(f"{base_url}/feed", timeout=10).text
    print("/feed（無限捲動）：requests 找到", len(BeautifulSoup(feed_html, "html.parser").select("article.post")), "篇")

    more_html = requests.get(f"{base_url}/load-more", timeout=10).text
    more_soup = BeautifulSoup(more_html, "html.parser")
    print("/load-more（載入更多）：requests 找到", len(more_soup.select("article.post")), "篇，按鈕：", more_soup.select_one("#load-more").get_text())

    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))
    print("Selenium 打開 /feed：", post_count(driver), "篇")
    print("原因：文章是 JavaScript 在『打開網頁』與『捲動』時，才向網站要來的。")


## demo_03_execute_script


In [ ]:
def demo_03_execute_script(driver, base_url):
    title("03｜execute_script：直接對網頁下 JavaScript 指令")
    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))

    height = driver.execute_script("return document.body.scrollHeight")  # 有 return 才會把結果帶回 Python
    print("頁面總高度：", height, "px｜目前捲動位置：", driver.execute_script("return window.scrollY"), "px")

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")  # 沒有 return：只執行動作
    time.sleep(0.1)  # 只為了讓下一行印出的位置是捲完之後的（示範用，正式程式不要這樣寫）
    print("捲到底之後的位置：", driver.execute_script("return window.scrollY"), "px")
    print("提醒：Selenium 沒有『捲動』的指令，捲動是靠 execute_script 下 JavaScript。")


## demo_04_wait_for_more


In [ ]:
def demo_04_wait_for_more(driver, base_url):
    title("04｜捲到底之後，要等新文章出現")
    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))

    before = post_count(driver)
    scroll_to_bottom(driver)
    print("剛捲完馬上數：", post_count(driver), "篇（新文章還沒到，因為網站要 0.3 秒才回應）")

    # 自訂等待條件：lambda 接收 driver，回傳 True 就代表條件成立
    started = time.time()
    WebDriverWait(driver, 5).until(lambda d: post_count(d) > before)
    print(f"等了 {time.time() - started:.1f} 秒後：", post_count(driver), "篇")
    print("寫法：until(lambda d: 條件)。條件可以是任何你想等的事情，不只是元素出現。")


## load_all_by_growth


In [ ]:
def load_all_by_growth(driver, max_rounds=30):
    """做法一：一直捲，直到『等了 3 秒都沒有新文章』就當作到底了。通用，但最後會多等 3 秒。"""
    for _ in range(max_rounds):  # 一定要有上限，避免網頁永遠載不完時程式停不下來
        before = post_count(driver)
        scroll_to_bottom(driver)
        try:
            WebDriverWait(driver, 3).until(lambda d: post_count(d) > before)
        except TimeoutException:
            break
    return post_count(driver)


## load_all_by_marker


In [ ]:
def load_all_by_marker(driver, max_rounds=30):
    """做法二：網頁到底會出現『已載入全部』的提示（id 為 end-message），看到就停。最快。"""
    for _ in range(max_rounds):
        if driver.find_elements(By.ID, "end-message"):
            break
        before = post_count(driver)
        scroll_to_bottom(driver)
        WebDriverWait(driver, 5).until(lambda d: post_count(d) > before)
    return post_count(driver)


## demo_05_scroll_all


In [ ]:
def demo_05_scroll_all(driver, base_url):
    title("05｜捲到全部載入：兩種停止的方法")
    for name, loader in (("做法一：等不到新文章就停", load_all_by_growth), ("做法二：看到結尾提示就停", load_all_by_marker)):
        driver.get(f"{base_url}/feed")
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))
        started = time.time()
        total = loader(driver)
        print(f"{name}：共 {total} 篇，花了 {time.time() - started:.1f} 秒")
    print("結尾提示：", driver.find_element(By.ID, "end-message").text)
    print("重點：迴圈一定要有『停止條件』與『次數上限』，否則遇到載不完的網頁會永遠停不下來。")


## demo_06_stop_early


In [ ]:
def demo_06_stop_early(driver, base_url):
    title("06｜不用全部載入：找到目標就停")
    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))

    target_id = 35
    target = f"article.post[data-id='{target_id}']"
    for _ in range(30):
        if driver.find_elements(By.CSS_SELECTOR, target) or driver.find_elements(By.ID, "end-message"):
            break
        before = post_count(driver)
        scroll_to_bottom(driver)
        WebDriverWait(driver, 5).until(lambda d: post_count(d) > before)

    found = driver.find_elements(By.CSS_SELECTOR, target)
    print("找到了嗎：", bool(found), "｜標題：", found[0].find_element(By.CSS_SELECTOR, "h3").text)
    print("停下來時只載入了", post_count(driver), "篇（不是全部 60 篇）→ 少載入就少請求，對網站也比較客氣。")


## demo_07_load_more_button


In [ ]:
def demo_07_load_more_button(driver, base_url):
    title("07｜按鈕式：一直按「載入更多」")
    driver.get(f"{base_url}/load-more")
    print("剛打開：", post_count(driver), "篇")

    clicks = 0
    while driver.find_elements(By.ID, "load-more"):  # 按鈕還在，就代表還有資料
        before = post_count(driver)
        driver.find_element(By.ID, "load-more").click()
        WebDriverWait(driver, 5).until(lambda d: post_count(d) > before)
        clicks += 1
        print(f"  按第 {clicks} 次：{post_count(driver)} 篇")
    print("按鈕消失了，總共", post_count(driver), "篇")
    print("停止條件就是『按鈕不見了』；用 find_elements（複數）檢查，找不到會得到空 list，不會出錯。")


## demo_08_xpath


In [ ]:
def demo_08_xpath(driver, base_url):
    title("08｜XPath 入門：用文字找元素")
    driver.get(f"{base_url}/search")

    print("//li                          所有 li：", len(driver.find_elements(By.XPATH, "//li")))
    print("//li[@class='result']         屬性等於：", len(driver.find_elements(By.XPATH, "//li[@class='result']")))
    print("//span[text()='進階']         文字完全等於：", len(driver.find_elements(By.XPATH, "//span[text()='進階']")))
    print("//li[contains(., 'Python')]   文字包含：", len(driver.find_elements(By.XPATH, "//li[contains(., 'Python')]")))

    parents = driver.find_elements(By.XPATH, "//span[text()='進階']/..")  # /.. 是往上找父層
    print("//span[text()='進階']/..      父層 li：", [li.text for li in parents])
    titles = driver.find_elements(By.XPATH, "//span[text()='進階']/../span[@class='title']")
    print("再往下找標題：", [item.text for item in titles])

    driver.get(f"{base_url}/load-more")
    button = driver.find_element(By.XPATH, "//button[text()='載入更多']")
    print("用按鈕上的字找到按鈕：", button.get_attribute("id"))

    print("\nCSS 與 XPath 怎麼選？")
    print("  能用 CSS 就用 CSS（短、你已經會）。")
    print("  要『依文字找』或『從子元素往上找父層』時，才用 XPath。")


## demo_09_beautifulsoup_pandas


In [ ]:
def demo_09_beautifulsoup_pandas(driver, base_url):
    title("09｜渲染完成後，交給 BeautifulSoup 與 pandas")
    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))
    load_all_by_marker(driver)

    posts = parse_posts(driver.page_source)
    df = pd.DataFrame(posts)
    print("形狀：", df.shape, "｜id 沒有重複：", df["id"].is_unique)
    print(df.head(3).to_string(index=False))
    print("\n各標籤的文章數與閱讀數：")
    print(df.groupby("tag").agg(posts=("id", "count"), views=("views", "sum")).to_string())

    Path("output").mkdir(exist_ok=True)
    df.to_csv("output/feed.csv", index=False, encoding="utf-8-sig")
    print("\n已存檔：output/feed.csv")

    snapshot = pd.DataFrame(parse_posts(Path("fixtures/feed_rendered.html").read_text(encoding="utf-8")))
    print("存好的『捲完之後』快照 fixtures/feed_rendered.html 解析結果和剛剛一樣：", snapshot.equals(df))
    print("（Chrome 出狀況時，可以直接拿這份快照做後面的 pandas 練習。）")


## fetch_all_posts_by_api


In [ ]:
def fetch_all_posts_by_api(base_url):
    """直接呼叫網頁背後的 API，一頁一頁拿，不需要瀏覽器。"""
    posts, offset = [], 0
    while offset is not None:
        response = requests.get(f"{base_url}/api/feed", params={"offset": offset, "limit": 10}, timeout=10)
        data = response.json()
        posts.extend(data["items"])
        offset = data["next_offset"]  # 最後一頁是 None
        time.sleep(0.05)  # 真實網站請改成 1 秒以上
    return posts


## demo_10_find_api


In [ ]:
def demo_10_find_api(driver, base_url):
    title("10｜更好的辦法：找出網頁背後的 API")
    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))
    scroll_to_bottom(driver)
    WebDriverWait(driver, 5).until(lambda d: post_count(d) > 10)

    # 不用開 F12：直接問瀏覽器，這個網頁向哪些網址要過資料
    entries = driver.execute_script("return performance.getEntriesByType('resource').map(e => e.name)")
    api_urls = [entry.replace(base_url, "") for entry in entries if "/api/" in entry]
    print("網頁呼叫過的 API：", api_urls)

    started = time.time()
    api_posts = fetch_all_posts_by_api(base_url)
    api_seconds = time.time() - started

    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))
    started = time.time()
    load_all_by_marker(driver)
    selenium_posts = parse_posts(driver.page_source)
    selenium_seconds = time.time() - started

    print(f"Selenium 捲動：{len(selenium_posts)} 篇，約 {selenium_seconds:.1f} 秒")
    print(f"直接呼叫 API：{len(api_posts)} 篇，約 {api_seconds:.1f} 秒")
    print("兩邊資料一樣：", pd.DataFrame(api_posts).equals(pd.DataFrame(selenium_posts)))
    print("→ 能找到 API 就用 requests（快、穩、不用開瀏覽器）；找不到、或要登入互動時，才用 Selenium。")


## demo_11_common_problems


In [ ]:
def demo_11_common_problems(driver, base_url):
    title("11｜常見問題")
    driver.get(f"{base_url}/feed")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.post")))

    # 問題一：另一種捲法：把最後一個元素捲進畫面
    last = driver.find_elements(By.CSS_SELECTOR, "article.post")[-1]
    driver.execute_script("arguments[0].scrollIntoView()", last)
    WebDriverWait(driver, 5).until(lambda d: post_count(d) > 10)
    print("1. scrollIntoView 也能觸發載入：", post_count(driver), "篇（arguments[0] 就是傳進去的元素）")

    # 問題二：資料是不是完整、有沒有重複
    load_all_by_marker(driver)
    expected_total = int("".join(ch for ch in driver.find_element(By.ID, "end-message").text if ch.isdigit()))
    ids = [post["id"] for post in parse_posts(driver.page_source)]
    print(f"2. 網頁說共 {expected_total} 篇，抓到 {len(ids)} 篇，沒有重複：{len(ids) == len(set(ids))}")

    print("3. 捲了卻沒有新資料？先確認：視窗是不是太小或沒有真的捲到底、新資料是否需要等待、網頁是不是改成按鈕式。")
    print("4. 迴圈停不下來？加上 max_rounds 次數上限，並用 print 看每一輪的數量有沒有增加。")
    print("5. 無限捲動的網頁可能真的有無限多筆（例如社群動態）：先決定要幾筆，達到就停。")


## main


In [ ]:
def main():
    selected = sorted(sys.argv[1:] or DEMOS)
    server, base_url = start_server()
    print("本機練習站：", base_url, "｜無頭模式：", HEADLESS)
    driver = None
    try:
        for key in selected:
            if key in DEMOS:
                if driver is None:
                    driver = new_driver()
                DEMOS[key](driver, base_url)
            else:
                print("沒有這個單元：", key)
    finally:
        if driver is not None:
            driver.quit()
        stop_server(server)
    print("\n全部完成。練習站與瀏覽器都已關閉。")


## 執行 demo_01_preview
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_01_preview(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_02_requests_cannot
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_02_requests_cannot(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_03_execute_script
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_03_execute_script(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_04_wait_for_more
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_04_wait_for_more(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_05_scroll_all
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_05_scroll_all(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_06_stop_early
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_06_stop_early(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_07_load_more_button
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_07_load_more_button(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_08_xpath
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_08_xpath(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_09_beautifulsoup_pandas
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_09_beautifulsoup_pandas(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_10_find_api
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_10_find_api(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_11_common_problems
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_11_common_problems(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)
